In [2]:
# ler excel
import pandas as pd

# folha Resumo
df_smh = pd.read_excel('diagnostico.xlsx', sheet_name='Resumo')
# Exibir as primeiras linhas do DataFrame
print(df_smh.head())

  Código do diagnóstico                                        Diagnóstico  \
0                  E785                        Hyperlipidemia, Unspecified   
1                   I10                   Essential (Primary) Hypertension   
2                 Z7984  Long Term (Current) Use Of Oral Hypoglycemic D...   
3                  E119     Type 2 Diabetes Mellitus Without Complications   
4                  E669                               Obesity, Unspecified   

   Qtd de episódios  
0              1599  
1              1580  
2               591  
3               538  
4               491  


In [3]:
df_icd10 = pd.read_excel('icd10cm.xlsx', sheet_name='ICD10CM')
print(df_icd10.head())

  Capitulo ICD-10-CM_ Código  \
0                    A00-B99   
1                    A00-B99   
2                    A00-B99   
3                    A00-B99   
4                    A00-B99   

                             Capitulo ICD-10-CM_desc  \
0  Certain infectious and parasitic diseases (A00...   
1  Certain infectious and parasitic diseases (A00...   
2  Certain infectious and parasitic diseases (A00...   
3  Certain infectious and parasitic diseases (A00...   
4  Certain infectious and parasitic diseases (A00...   

                   Capitulo ICD-10-CM_desc_PT Secção ICD-10-CM_Código  \
0  Algumas doenças infecciosas e parasitárias                 A00-A09   
1  Algumas doenças infecciosas e parasitárias                 A00-A09   
2  Algumas doenças infecciosas e parasitárias                 A00-A09   
3  Algumas doenças infecciosas e parasitárias                 A00-A09   
4  Algumas doenças infecciosas e parasitárias                 A00-A09   

            Secção ICD-10-CM_De

In [4]:
# filtrar df_id10 coluna Código para corresponder à coluna Diagnóstico em df_smh
filtered_icd10 = df_icd10[df_icd10['Código'].isin(df_smh['Código do diagnóstico'])]
print(len(filtered_icd10))

2097


In [5]:

# guardar 'Descrição PT_(Longa)', 'Capitulo ICD-10-CM_desc',      'Descrição PT_(Curta)','Código', 'Válido',
bd = filtered_icd10[['Código', 'Capitulo ICD-10-CM_desc_PT', 'Descrição PT_(Longa)','Secção ICD-10-CM_Desc_PT', 'Válido']]

# guardar em excel
print(len(bd))


2097


In [6]:
#  'Capitulo ICD-10-CM_desc_PT'  unique
capitulos = bd['Capitulo ICD-10-CM_desc_PT'].unique()

print(capitulos)

sections = bd['Secção ICD-10-CM_Desc_PT'].unique()

['Algumas doenças infecciosas e parasitárias' 'Neoplasias'
 'Doenças do sangue e dos órgãos hematopoéticos e alguns transtornos imunitários'
 'Doenças endócrinas, nutricionais e metabólicas'
 'Transtornos mentais, comportamentais e de neurodesenvolvimento'
 'Doenças do sistema nervoso' 'Doenças do olho e anexos'
 'Doenças do ouvido e da apófise mastóide'
 'Doenças do aparelho circulatório' 'Doenças do aparelho respiratório'
 'Doenças do aparelho digestivo' 'Doenças da pele e do tecido subcutâneo'
 'Doenças do aparelho osteomuscular e do tecido conjuntivo'
 'Doenças do aparelho geniturinário'
 'Malformações congénitas, deformações, anomalias cromossómicas, e doenças genéticas (Q00-QA0)'
 'Sintomas/sinais/achados anormais d exames clínicos/laboratoriais,ÑClassificadosNoutraParte'
 'Lesões, envenenamento e algumas outras consequências de causas externas'
 'Códigos para fins especiais (U00-U85)' 'Causas externas de morbilidade'
 'Fatores que influenciam o estado de saúde e o contacto com o

In [7]:
import pandas as pd
from sqlalchemy import create_engine, text

# =======================================
# 1) Conexão com o MySQL usando SQLAlchemy
# =======================================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# =======================================
# 2) Inserir dados da lista/Series sections
# =======================================

with engine.connect() as conn:
    for item in sections:   # se for DataFrame, troque para sections['coluna']
        conn.execute(
            text("INSERT INTO sections (name, kind) VALUES (:name, :kind)"),
            {"name": item, "kind": "diagnosticos"}
        )

    conn.commit()

print("Dados inseridos com sucesso!")


Dados inseridos com sucesso!


In [8]:
with engine.connect() as conn:
    for item in capitulos:   # se for DataFrame, troque para sections['coluna']
        conn.execute(
            text("INSERT INTO categories (name, kind) VALUES (:name, :kind)"),
            {"name": item, "kind": "diagnosticos"}
        )

    conn.commit()
print("Dados inseridos com sucesso!")

Dados inseridos com sucesso!


In [9]:

# Capitulo ICD-10-CM_desc_PT as categoria
bd = bd.rename(columns={'Capitulo ICD-10-CM_desc_PT': 'categoria', 'Secção ICD-10-CM_Desc_PT': 'secao', 'Código': 'codigo', 'Descrição PT_(Longa)': 'description', 'Válido': 'valid'})    
print(bd.head())



    codigo                                   categoria  \
47   A0471  Algumas doenças infecciosas e parasitárias   
48   A0472  Algumas doenças infecciosas e parasitárias   
94     A09  Algumas doenças infecciosas e parasitárias   
96    A150  Algumas doenças infecciosas e parasitárias   
128   A182  Algumas doenças infecciosas e parasitárias   

                                           description  \
47   Enterocolite devido a Clostridium Difficile, r...   
48   Enterocolite devido a Clostridium Difficile, n...   
94   Gastroenterite e colite infeciosa, sem outra e...   
96                                Tuberculose pulmonar   
128              Linfadenopatia tuberculosa periférica   

                               secao  valid  
47   Doenças infecciosas intestinais      1  
48   Doenças infecciosas intestinais      1  
94   Doenças infecciosas intestinais      1  
96                       Tuberculose      1  
128                      Tuberculose      1  


In [11]:
import pandas as pd
from sqlalchemy import create_engine, text

# ==========================
# 1) Criar engine (SQLite ou MySQL)
# ==========================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# ==========================
# 2) Preparar caches
# ==========================
cache_categoria = {}
cache_codigo = {}
cache_secao = {}

# ==========================
# 3) Buscar specialty_id apenas uma vez
# ==========================
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT id FROM specialties WHERE name = :name"),
        {"name": "Cirurgia Geral"}
    ).fetchone()
    specialty_id = result[0] if result else None

# ==========================
# 4) Preparar lista de inserções
# ==========================
insercoes = []

with engine.connect() as conn:
    for row in bd.itertuples():
        
        # ----- Categoria -----
        categoria = row.categoria
        if categoria not in cache_categoria:
            r = conn.execute(
                text("SELECT id FROM categories WHERE name = :name"),
                {"name": categoria}
            ).fetchone()
            cache_categoria[categoria] = r[0] if r else None
        categoria_id = cache_categoria[categoria]

        # ----- Código ICD -----
        codigo = row.codigo
        if codigo not in cache_codigo:
            r = conn.execute(
                text("SELECT id FROM icd10cms WHERE codigo = :codigo"),
                {"codigo": codigo}
            ).fetchone()
            cache_codigo[codigo] = r[0] if r else None
        codigo_id = cache_codigo[codigo]

        # ----- Seção -----
        secao = getattr(row, "secao", None)
        if secao:
            if secao not in cache_secao:
                r = conn.execute(
                    text("SELECT id FROM sections WHERE name = :name"),
                    {"name": secao}
                ).fetchone()
                cache_secao[secao] = r[0] if r else None
            secao_id = cache_secao[secao]
        else:
            secao_id = None

        # ----- Preparar inserção -----
        insercoes.append({
            "tabela_origem": "icd10cms",
            "codigo_id": codigo_id,
            "category_id": categoria_id,
            "specialty_id": specialty_id,
            "section_id": secao_id
        })

# ==========================
# 5) Inserção em massa usando executemany
# ==========================
with engine.connect() as conn:
    conn.execute(
        text("""
            INSERT INTO favoritos (tabela_origem, codigo_id, category_id, specialty_id, section_id)
            VALUES (:tabela_origem, :codigo_id, :category_id, :specialty_id, :section_id)
        """),
        insercoes
    )
    conn.commit()

print("Inserção concluída com sucesso!")


Inserção concluída com sucesso!
